# Week 61, Day 6: Forge: Week 61 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w061/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
p = np.full(20, 1 / 20)
print("1.", round(float(np.exp(-(p * np.log(p)).sum())), 3))
print("2.", 100 * 0.6 / (0.6 + 1.4), "%")
u = np.linspace(-np.sqrt(3), np.sqrt(3), 200001)
print("3.", round(float(np.mean(u ** 4) - 3), 3), "(exactly -1.2)")

### Exercise: Part A: PCA against factor analysis for decoding

Fit factor analysis with three factors on the 400 training trials and compute the posterior means E[z | x] = beta (x - mu) for training and test trials. Fit a linear regression from each three-dimensional summary to speed on the training trials and compute R squared on the test trials.

In [ ]:
import numpy as np
gA = np.random.default_rng(616)
n, d = 600, 30
speed = gA.normal(size=n)
Zlat = np.column_stack([speed, gA.normal(size=(n, 2))])
L_true = gA.normal(size=(d, 3))
psi = gA.uniform(0.2, 1.0, d); psi[gA.choice(d, 6, replace=False)] = 15.0       # six very noisy neurons
X = Zlat @ L_true.T + gA.normal(size=(n, d)) * np.sqrt(psi)
tr, te = np.arange(400), np.arange(400, 600)

def fa_em(X, q, iters=500):
    n, d = X.shape; Xc = X - X.mean(0); S = Xc.T @ Xc / n
    L = np.random.default_rng(0).normal(size=(d, q)); Psi = np.diag(S).copy()
    for _ in range(iters):
        beta = L.T @ np.linalg.inv(L @ L.T + np.diag(Psi))
        Ezz = np.eye(q) - beta @ L + beta @ S @ beta.T
        L = S @ beta.T @ np.linalg.inv(Ezz)
        Psi = np.maximum(np.diag(S - L @ beta @ S), 1e-6)
    return L, Psi

def r2_linear(Ftr, ytr, Fte, yte):
    A = np.column_stack([np.ones(len(Ftr)), Ftr]); w = np.linalg.lstsq(A, ytr, rcond=None)[0]
    p = np.column_stack([np.ones(len(Fte)), Fte]) @ w
    return 1 - np.mean((yte - p) ** 2) / np.var(yte)

mu = X[tr].mean(0)
Vt = np.linalg.svd(X[tr] - mu, full_matrices=False)[2][:3]
r2_pca = r2_linear((X[tr] - mu) @ Vt.T, speed[tr], (X[te] - mu) @ Vt.T, speed[te])
L_fa, Psi_fa = fa_em(X[tr], 3)
beta = ...
Ztr, Zte = ..., ...
r2_fa = r2_linear(Ztr, speed[tr], Zte, speed[te])
print(f"decoding speed from 3 dimensions, test R squared: PCA {r2_pca:.3f}, factor analysis {r2_fa:.3f}")

In [ ]:
import numpy as np
def test_decode():
    "speed is decoded better from the factor-analysis latents than from the principal components"
    assert r2_fa > r2_pca + 0.05 and r2_fa > 0.7

test_decode()
print("All checks passed")

### Exercise: Part B: rank by held-out entries

Complete masked_nmf, then for ranks 1 to 7 fit it with 10% of entries hidden and record the mean squared error on the hidden entries. Choose the smallest rank whose error is within 15% of the lowest.

In [ ]:
gB = np.random.default_rng(6161)
W0 = gB.exponential(1.0, (200, 4)) * (gB.random((200, 4)) < 0.3)
H0 = gB.exponential(1.0, (4, 60))
Xnmf = W0 @ H0 + 0.3 * gB.exponential(1.0, (200, 60))
mask = gB.random(Xnmf.shape) > 0.1                     # True where visible

def masked_nmf(X, M, k, seed, iters=500):
    g = np.random.default_rng(seed)
    W = g.uniform(0.1, 1, (X.shape[0], k)); H = g.uniform(0.1, 1, (k, X.shape[1]))
    MX = M * X
    for _ in range(iters):
        H *= (W.T @ MX) / (W.T @ (M * (W @ H)) + 1e-12)
        W *= ...
    return W, H

heldout_err = {}
for k in range(1, 8):
    W, H = masked_nmf(Xnmf, mask, k, seed=k)
    heldout_err[k] = np.mean((Xnmf - W @ H)[~mask] ** 2)
lowest = min(heldout_err.values())
best_rank = min(k for k in heldout_err if heldout_err[k] <= 1.15 * lowest)   # smallest rank within 15% of the best
print("held-out error by rank:", {k: round(float(v), 4) for k, v in heldout_err.items()}, "-> rank", best_rank)

In [ ]:
import numpy as np
def test_rank():
    "the parsimonious choice by held-out error is the true rank, 4"
    assert best_rank == 4

test_rank()
print("All checks passed")